# 🧠 ARC-AGI-2: SOTA 33.89+ Verified Cognitive Solver

<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 100%); padding: 22px; border-radius: 12px; border: 1px solid #30363d; margin-bottom: 20px; box-shadow: 0 4px 14px rgba(0,0,0,0.4);">
    <h2 style="color: #58a6ff; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;">
        🚀 SOTA Proven Architecture: Minimal GPU-Logsumexp Beam Search + Certified Geometric Invariant Synthesis (33.89+)
    </h2>
    <p style="color: #c9d1d9; font-size: 15px; line-height: 1.6;">
        A high-performance, monotonic solution for <b>ARC Prize 2026 — ARC-AGI-2</b> built directly upon the official <b>33.89 LB SOTA baseline</b>.
        Guarantees <b>monotonic coverage ($\ge 33.89$)</b> by preserving 100% of the proven neural pipeline (128 TTT sequences, LoRA rank 256, 16 inference views across 4x L4 GPUs, GPU-side 12-token logsumexp)
        while augmenting empty/duplicate attempt slots with <b>System 2 Demonstration-Certified Geometric Program Synthesis</b>.
    </p>
    <div style="display: flex; gap: 12px; flex-wrap: wrap; margin-top: 12px;">
        <span style="background: #238636; color: #ffffff; padding: 4px 10px; border-radius: 6px; font-size: 13px; font-weight: 600;">🏆 Target SOTA: 33.89+ LB</span>
        <span style="background: #1f6feb; color: #ffffff; padding: 4px 10px; border-radius: 6px; font-size: 13px; font-weight: 600;">⚡ 4x L4 Work-Stealing Queue</span>
        <span style="background: #8957e5; color: #ffffff; padding: 4px 10px; border-radius: 6px; font-size: 13px; font-weight: 600;">🛡️ 100% Demonstration Certified</span>
        <span style="background: #da3633; color: #ffffff; padding: 4px 10px; border-radius: 6px; font-size: 13px; font-weight: 600;">📈 Strictly Monotonic</span>
    </div>
</div>

### 🔬 Core Foundations & Guaranteed Invariants
1. **Pristine 33.89 LB Neural Foundation**:
   - Preserves the exact 128-sequence TTT training loop, LoRA parameters, and 16 augmented inference views (4 batches of 4) that achieved 33.89 LB.
   - Preserves the GPU-side `turbo_dfs` logsumexp patch (indexing only the 12 ARC tokens on device) and GPU target gathering in `calc_scores`.
   - Preserves the dynamic `mp.Manager().Queue()` work-stealing queue across all 4 L4 GPUs for optimal multi-GPU load balancing.
2. **System 2: Demonstration-Certified Geometric Program Synthesis (`arc2_symbolic.py`)**:
   - Solves exact geometric and topological invariants: masked reflections, obstacle border rays, periodic tiling, and component selectors.
   - Any program matching 100% of demonstration pairs is certified.
3. **Strict Monotonic Injection**:
   - Every single valid neural candidate generated by the 33.89 baseline in `attempt_1` is **strictly preserved**.
   - Certified geometric programs fill empty timeout slots in `attempt_1` or replace redundant duplicate/dummy `[[0]]` slots in `attempt_2`.
   - Distinctness guarantee: `attempt_1` and `attempt_2` are guaranteed to be distinct, non-empty, valid grids.


In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600
print(f"[Timer] Set global deadline: {global_end_time:.1f} (11.83 hours)")


In [ ]:
# Preserve the baseline environment workaround to eliminate GPU memory contention.
!pip uninstall -y tensorflow


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** skipping unreadable result file {key}: {e!r}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc2_symbolic.py
from __future__ import annotations

import json
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np


Grid = Tuple[Tuple[int, ...], ...]


def to_arr(grid) -> np.ndarray:
    return np.asarray(grid, dtype=np.int8)


def to_grid(arr: np.ndarray) -> Grid:
    arr = np.asarray(arr, dtype=np.int8)
    return tuple(tuple(int(x) for x in row) for row in arr.tolist())


def valid_grid(arr: np.ndarray) -> bool:
    arr = np.asarray(arr)
    return arr.ndim == 2 and 1 <= arr.shape[0] <= 30 and 1 <= arr.shape[1] <= 30


def bg_color(a: np.ndarray) -> int:
    vals, cnt = np.unique(a, return_counts=True)
    return int(vals[int(np.argmax(cnt))])


def bbox_non_bg(a: np.ndarray, bg: Optional[int] = None) -> Optional[Tuple[int, int, int, int]]:
    if bg is None:
        bg = bg_color(a)
    ys, xs = np.where(a != bg)
    if len(ys) == 0:
        return None
    return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1


def crop_bbox(a: np.ndarray, bg: Optional[int] = None) -> np.ndarray:
    bb = bbox_non_bg(a, bg)
    if bb is None:
        return a.copy()
    y0, y1, x0, x1 = bb
    return a[y0:y1, x0:x1].copy()


def trim_uniform_border(a: np.ndarray) -> np.ndarray:
    out = a.copy()
    while out.shape[0] > 1 and len(set(out[0].tolist())) == 1:
        out = out[1:]
    while out.shape[0] > 1 and len(set(out[-1].tolist())) == 1:
        out = out[:-1]
    while out.shape[1] > 1 and len(set(out[:, 0].tolist())) == 1:
        out = out[:, 1:]
    while out.shape[1] > 1 and len(set(out[:, -1].tolist())) == 1:
        out = out[:, :-1]
    return out.copy()


def connected_components(a: np.ndarray, include_bg: bool = False):
    h, w = a.shape
    bg = bg_color(a)
    seen = np.zeros((h, w), dtype=bool)
    comps = []
    for y in range(h):
        for x in range(w):
            if seen[y, x] or (not include_bg and int(a[y, x]) == bg):
                continue
            col = int(a[y, x])
            q = deque([(y, x)])
            seen[y, x] = True
            cells = []
            while q:
                cy, cx = q.popleft()
                cells.append((cy, cx))
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = cy + dy, cx + dx
                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and int(a[ny, nx]) == col:
                        seen[ny, nx] = True
                        q.append((ny, nx))
            ys = [p[0] for p in cells]
            xs = [p[1] for p in cells]
            comps.append(
                {
                    "color": col,
                    "cells": cells,
                    "area": len(cells),
                    "bbox": (min(ys), max(ys) + 1, min(xs), max(xs) + 1),
                    "height": max(ys) - min(ys) + 1,
                    "width": max(xs) - min(xs) + 1,
                }
            )
    return comps


def crop_component(a: np.ndarray, comp: dict, mask_bg: Optional[int] = None) -> np.ndarray:
    y0, y1, x0, x1 = comp["bbox"]
    sub = a[y0:y1, x0:x1].copy()
    if mask_bg is not None:
        out = np.full_like(sub, int(mask_bg))
        for y, x in comp["cells"]:
            out[y - y0, x - x0] = a[y, x]
        return out
    return sub


def transforms() -> List[Tuple[str, Callable[[np.ndarray], np.ndarray]]]:
    return [
        ("id", lambda a: a.copy()),
        ("rot90", lambda a: np.rot90(a, 1).copy()),
        ("rot180", lambda a: np.rot90(a, 2).copy()),
        ("rot270", lambda a: np.rot90(a, 3).copy()),
        ("flipud", lambda a: np.flipud(a).copy()),
        ("fliplr", lambda a: np.fliplr(a).copy()),
        ("transpose", lambda a: a.T.copy()),
        ("anti_transpose", lambda a: np.fliplr(np.flipud(a)).T.copy()),
    ]


def tile_to_shape(a: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    reps = (int(np.ceil(shape[0] / a.shape[0])), int(np.ceil(shape[1] / a.shape[1])))
    return np.tile(a, reps)[: shape[0], : shape[1]].copy()


def shifted_tile_to_shape(a: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    h, w = a.shape
    out = np.zeros(shape, dtype=np.int8)
    for y in range(shape[0]):
        shift = y // h
        for x in range(shape[1]):
            out[y, x] = a[y % h, (x + shift) % w]
    return out


def infer_color_map(srcs: Sequence[np.ndarray], dsts: Sequence[np.ndarray]) -> Optional[Dict[int, int]]:
    mapping: Dict[int, int] = {}
    for s, d in zip(srcs, dsts):
        if s.shape != d.shape:
            return None
        for x, y in zip(s.ravel().tolist(), d.ravel().tolist()):
            x = int(x)
            y = int(y)
            if x in mapping and mapping[x] != y:
                return None
            mapping[x] = y
    return mapping


def apply_color_map(a: np.ndarray, mapping: Dict[int, int], default_keep: bool = True) -> np.ndarray:
    out = a.copy()
    for c in range(10):
        if c in mapping:
            out[a == c] = int(mapping[c])
        elif not default_keep:
            out[a == c] = 0
    return out


def remap_by_counts(src: np.ndarray, dst: np.ndarray) -> Optional[Dict[int, int]]:
    """Fallback color map by rank of color frequencies."""
    sc = Counter(src.ravel().tolist())
    dc = Counter(dst.ravel().tolist())
    if len(sc) != len(dc):
        return None
    s_order = [c for c, _ in sorted(sc.items(), key=lambda kv: (-kv[1], kv[0]))]
    d_order = [c for c, _ in sorted(dc.items(), key=lambda kv: (-kv[1], kv[0]))]
    return {int(s): int(d) for s, d in zip(s_order, d_order)}


@dataclass
class Candidate:
    name: str
    outputs: List[np.ndarray]
    score: float


class Arc2SymbolicSolver:
    def __init__(self):
        self.max_candidates = 64

    def solve_task(self, task: dict) -> List[List[np.ndarray]]:
        tests = [to_arr(x["input"]) for x in task["test"]]
        by_test = [[] for _ in tests]
        for cand in self._exact_programs(task):
            for i, out in enumerate(cand.outputs):
                self._add(by_test[i], out, cand.score, cand.name)
        for cand in self._heuristic_programs(task):
            for i, out in enumerate(cand.outputs):
                self._add(by_test[i], out, cand.score, cand.name)
        answers = []
        for i, inp in enumerate(tests):
            if not by_test[i]:
                by_test[i].append((0.0, "fallback_input", inp.copy()))
            ordered = []
            seen = set()
            for _score, _name, out in sorted(by_test[i], key=lambda x: (-x[0], x[1])):
                if not valid_grid(out):
                    continue
                key = to_grid(out)
                if key not in seen:
                    seen.add(key)
                    ordered.append(out.astype(np.int8, copy=False))
                if len(ordered) == 2:
                    break
            while len(ordered) < 2:
                ordered.append(ordered[0].copy())
            answers.append(ordered[:2])
        return answers

    def _add(self, bucket, out: np.ndarray, score: float, name: str):
        if valid_grid(out):
            bucket.append((float(score), name, out.astype(np.int8, copy=False)))

    def _pairs(self, task: dict):
        return [(to_arr(p["input"]), to_arr(p["output"])) for p in task["train"]]

    def _exact_programs(self, task: dict) -> Iterable[Candidate]:
        pairs = self._pairs(task)
        tests = [to_arr(x["input"]) for x in task["test"]]
        yield from self._fit_border_ray_obstacle(task)
        yield from self._fit_masked_reflection(task)
        yield from self._fit_constant_output(pairs, tests)
        yield from self._fit_unary_transform(pairs, tests)
        yield from self._fit_crop_transform(pairs, tests)
        yield from self._fit_scale_transform(pairs, tests)
        yield from self._fit_periodic_tile(pairs, tests)
        yield from self._fit_shifted_tile(pairs, tests)
        yield from self._fit_masked_self_tile(pairs, tests)
        yield from self._fit_separator_overlay(pairs, tests)
        yield from self._fit_component_selector(pairs, tests)
        yield from self._fit_gridline_decomposition(pairs, tests)

    def _heuristic_programs(self, task: dict) -> Iterable[Candidate]:
        pairs = self._pairs(task)
        tests = [to_arr(x["input"]) for x in task["test"]]
        for idx, t in enumerate(tests):
            yield Candidate("input", [t.copy() if j == idx else tests[j].copy() for j in range(len(tests))], 0.05)
            yield Candidate("crop_non_bg", [crop_bbox(x) for x in tests], 0.10)
            yield Candidate("trim_border", [trim_uniform_border(x) for x in tests], 0.08)
        yield from self._nearest_shape_output(pairs, tests)


    def _simulate_border_rays(self, source, background, ray_color, obstacle_colors):
        grid = source.copy()
        height, width = grid.shape
        active = {
            col
            for col, value in enumerate(source[0])
            if int(value) == int(ray_color)
        }
        if not active:
            return None

        collisions = 0
        for row in range(1, height):
            runs = []
            col = 0
            while col < width:
                if int(source[row, col]) not in obstacle_colors:
                    col += 1
                    continue
                left = col
                while (
                    col + 1 < width
                    and int(source[row, col + 1]) in obstacle_colors
                ):
                    col += 1
                runs.append((left, col))
                col += 1

            blocked = set()
            spawned = set()
            for left, right in runs:
                hit = {
                    col
                    for col in active
                    if left <= col <= right
                }
                if not hit:
                    continue
                collisions += 1
                blocked.update(hit)

                for bypass_col in range(
                    max(0, left - 1),
                    min(width - 1, right + 1) + 1,
                ):
                    if int(grid[row - 1, bypass_col]) == int(background):
                        grid[row - 1, bypass_col] = ray_color

                for bypass_col in (left - 1, right + 1):
                    if 0 <= bypass_col < width:
                        spawned.add(bypass_col)
                        if int(grid[row, bypass_col]) == int(background):
                            grid[row, bypass_col] = ray_color

            active = (active - blocked) | spawned
            for active_col in active:
                if int(grid[row, active_col]) == int(background):
                    grid[row, active_col] = ray_color

        if collisions == 0:
            return None
        return grid

    def _fit_border_ray_obstacle(self, task):
        pairs = self._pairs(task)
        tests = [to_arr(item["input"]) for item in task["test"]]
        if not pairs or any(source.shape != target.shape for source, target in pairs):
            return

        transitions = {
            (int(source_value), int(target_value))
            for source, target in pairs
            for source_value, target_value in zip(source.ravel(), target.ravel())
            if int(source_value) != int(target_value)
        }
        if len(transitions) != 1:
            return
        background, ray_color = next(iter(transitions))
        if background == ray_color:
            return

        for source, target in pairs:
            allowed_change = (
                (source == background)
                & (target == ray_color)
            )
            if np.any((source != target) & ~allowed_change):
                return

        all_colors = {
            int(value)
            for source, _target in pairs
            for value in np.unique(source)
        }
        obstacle_colors = all_colors - {background, ray_color}
        if not obstacle_colors:
            return

        for rotation in range(4):
            valid = True
            for source, target in pairs:
                normalized = np.rot90(source, rotation)
                prediction = self._simulate_border_rays(
                    normalized,
                    background,
                    ray_color,
                    obstacle_colors,
                )
                if prediction is None:
                    valid = False
                    break
                prediction = np.rot90(prediction, -rotation)
                if not np.array_equal(prediction, target):
                    valid = False
                    break
            if not valid:
                continue

            outputs = []
            for test in tests:
                prediction = self._simulate_border_rays(
                    np.rot90(test, rotation),
                    background,
                    ray_color,
                    obstacle_colors,
                )
                if prediction is None:
                    valid = False
                    break
                outputs.append(np.rot90(prediction, -rotation))
            if valid:
                yield Candidate(
                    f"verified_border_rays:r{rotation}",
                    outputs,
                    1.20,
                )

    def _solid_color_rectangle(self, source, color):
        rows, cols = np.where(source == color)
        if len(rows) == 0:
            return None
        top = int(rows.min())
        bottom = int(rows.max()) + 1
        left = int(cols.min())
        right = int(cols.max()) + 1
        if (bottom - top) * (right - left) != len(rows):
            return None
        if not np.all(source[top:bottom, left:right] == color):
            return None
        return top, bottom, left, right

    def _recover_masked_reflection(self, source, mask_color):
        rectangle = self._solid_color_rectangle(source, mask_color)
        if rectangle is None:
            return None
        top, bottom, left, right = rectangle
        output_height = bottom - top
        output_width = right - left
        height, width = source.shape
        non_mask_count = int(np.count_nonzero(source != mask_color))
        minimum_support = max(20, (non_mask_count + 1) // 2)

        reflection_maps = []
        for orientation, size in (("horizontal", height), ("vertical", width)):
            for doubled_axis in range(2 * size - 1):
                matching = 0
                support = 0
                if orientation == "horizontal":
                    for row in range(height):
                        reflected_row = doubled_axis - row
                        if not 0 <= reflected_row < height:
                            continue
                        valid = (
                            (source[row, :] != mask_color)
                            & (source[reflected_row, :] != mask_color)
                        )
                        support += int(np.count_nonzero(valid))
                        matching += int(
                            np.count_nonzero(
                                (source[row, :] == source[reflected_row, :])
                                & valid
                            )
                        )
                else:
                    for col in range(width):
                        reflected_col = doubled_axis - col
                        if not 0 <= reflected_col < width:
                            continue
                        valid = (
                            (source[:, col] != mask_color)
                            & (source[:, reflected_col] != mask_color)
                        )
                        support += int(np.count_nonzero(valid))
                        matching += int(
                            np.count_nonzero(
                                (source[:, col] == source[:, reflected_col])
                                & valid
                            )
                        )
                if support >= minimum_support and matching == support:
                    reflection_maps.append((orientation, doubled_axis))

        if not reflection_maps:
            return None

        partial = np.full(
            (output_height, output_width),
            -1,
            dtype=np.int16,
        )
        for out_row in range(output_height):
            for out_col in range(output_width):
                row = top + out_row
                col = left + out_col
                reflected_values = []
                for orientation, doubled_axis in reflection_maps:
                    if orientation == "horizontal":
                        source_row = doubled_axis - row
                        source_col = col
                    else:
                        source_row = row
                        source_col = doubled_axis - col
                    if (
                        0 <= source_row < height
                        and 0 <= source_col < width
                        and int(source[source_row, source_col]) != int(mask_color)
                    ):
                        reflected_values.append(
                            int(source[source_row, source_col])
                        )
                if reflected_values:
                    if len(set(reflected_values)) != 1:
                        return None
                    partial[out_row, out_col] = reflected_values[0]

        known = partial >= 0
        if int(np.count_nonzero(known)) < max(
            4,
            (output_height * output_width + 2) // 3,
        ):
            return None

        candidates = {}
        source_shapes = {
            (output_height, output_width),
            (output_width, output_height),
        }
        for patch_height, patch_width in source_shapes:
            if patch_height > height or patch_width > width:
                continue
            for row in range(height - patch_height + 1):
                for col in range(width - patch_width + 1):
                    patch = source[
                        row:row + patch_height,
                        col:col + patch_width,
                    ]
                    if np.any(patch == mask_color):
                        continue
                    variants = (
                        patch,
                        np.rot90(patch, 1),
                        np.rot90(patch, 2),
                        np.rot90(patch, 3),
                        np.flipud(patch),
                        np.fliplr(patch),
                        patch.T,
                        np.flipud(np.fliplr(patch)).T,
                    )
                    for variant in variants:
                        if variant.shape != partial.shape:
                            continue
                        if not np.array_equal(variant[known], partial[known]):
                            continue
                        key = to_grid(variant)
                        candidates[key] = variant.copy()

        if len(candidates) != 1:
            return None
        return next(iter(candidates.values()))

    def _fit_masked_reflection(self, task):
        pairs = self._pairs(task)
        tests = [to_arr(item["input"]) for item in task["test"]]
        if not pairs:
            return

        shared_mask_colors = None
        for source, target in pairs:
            output_colors = {int(value) for value in np.unique(target)}
            pair_mask_colors = set()
            for value in np.unique(source):
                color = int(value)
                if color in output_colors:
                    continue
                rectangle = self._solid_color_rectangle(source, color)
                if rectangle is None:
                    continue
                top, bottom, left, right = rectangle
                if (bottom - top, right - left) == target.shape:
                    pair_mask_colors.add(color)
            if shared_mask_colors is None:
                shared_mask_colors = pair_mask_colors
            else:
                shared_mask_colors &= pair_mask_colors
        if not shared_mask_colors:
            return

        for mask_color in sorted(shared_mask_colors):
            if not all(
                (
                    prediction := self._recover_masked_reflection(
                        source,
                        mask_color,
                    )
                ) is not None
                and np.array_equal(prediction, target)
                for source, target in pairs
            ):
                continue

            outputs = []
            valid = True
            for test in tests:
                prediction = self._recover_masked_reflection(
                    test,
                    mask_color,
                )
                if prediction is None:
                    valid = False
                    break
                outputs.append(prediction)
            if valid:
                yield Candidate(
                    f"verified_masked_reflection:c{mask_color}",
                    outputs,
                    1.20,
                )

    def solve_verified_geometry(self, task):
        programs = (
            list(self._fit_border_ray_obstacle(task))
            + list(self._fit_masked_reflection(task))
        )
        if not programs:
            return None

        reference = programs[0].outputs
        for program in programs[1:]:
            if len(program.outputs) != len(reference):
                return None
            if any(
                not np.array_equal(left, right)
                for left, right in zip(program.outputs, reference)
            ):
                return None
        names = ",".join(program.name for program in programs)
        return names, reference

    def _fit_constant_output(self, pairs, tests):
        outs = [to_grid(o) for _, o in pairs]
        if len(set(outs)) == 1:
            out = to_arr(outs[0])
            yield Candidate("constant_output", [out.copy() for _ in tests], 0.96)

    def _fit_unary_transform(self, pairs, tests):
        for name, fn in transforms():
            srcs, dsts = [], []
            ok = True
            for x, y in pairs:
                z = fn(x)
                if z.shape != y.shape:
                    ok = False
                    break
                srcs.append(z)
                dsts.append(y)
            if ok:
                cmap = infer_color_map(srcs, dsts)
                if cmap is not None:
                    yield Candidate("unary:" + name, [apply_color_map(fn(t), cmap) for t in tests], 1.00)
            if all(fn(x).shape == y.shape for x, y in pairs):
                maps = [remap_by_counts(fn(x), y) for x, y in pairs]
                if all(m is not None for m in maps) and len({tuple(sorted(m.items())) for m in maps}) == 1:
                    cmap = maps[0]
                    yield Candidate("unary_countmap:" + name, [apply_color_map(fn(t), cmap) for t in tests], 0.74)

    def _fit_crop_transform(self, pairs, tests):
        crop_fns = [
            ("crop_bg", lambda a: crop_bbox(a, bg_color(a))),
            ("crop_zero", lambda a: crop_bbox(a, 0)),
            ("trim_uniform_border", trim_uniform_border),
        ]
        for crop_name, crop_fn in crop_fns:
            for tname, tfn in transforms():
                srcs, dsts = [], []
                ok = True
                for x, y in pairs:
                    try:
                        z = tfn(crop_fn(x))
                    except Exception:
                        ok = False
                        break
                    if z.shape != y.shape:
                        ok = False
                        break
                    srcs.append(z)
                    dsts.append(y)
                if not ok:
                    continue
                cmap = infer_color_map(srcs, dsts)
                if cmap is not None:
                    outs = [apply_color_map(tfn(crop_fn(t)), cmap) for t in tests]
                    yield Candidate(f"crop:{crop_name}:{tname}", outs, 0.98)

    def _fit_scale_transform(self, pairs, tests):
        for tname, tfn in transforms():
            factors = []
            srcs, dsts = [], []
            ok = True
            for x, y in pairs:
                z = tfn(x)
                if y.shape[0] % z.shape[0] or y.shape[1] % z.shape[1]:
                    ok = False
                    break
                fy, fx = y.shape[0] // z.shape[0], y.shape[1] // z.shape[1]
                if fy < 1 or fx < 1 or fy > 10 or fx > 10:
                    ok = False
                    break
                zz = np.repeat(np.repeat(z, fy, axis=0), fx, axis=1)
                if zz.shape != y.shape:
                    ok = False
                    break
                factors.append((fy, fx))
                srcs.append(zz)
                dsts.append(y)
            if ok and len(set(factors)) == 1:
                cmap = infer_color_map(srcs, dsts)
                if cmap is not None:
                    fy, fx = factors[0]
                    outs = []
                    for t in tests:
                        z = tfn(t)
                        z = np.repeat(np.repeat(z, fy, axis=0), fx, axis=1)
                        outs.append(apply_color_map(z, cmap))
                    yield Candidate(f"scale:{tname}:{fy}x{fx}", outs, 0.99)

    def _fit_periodic_tile(self, pairs, tests):
        for tname, tfn in transforms():
            for mode in ("factor", "constant"):
                srcs, dsts, shapes, factors = [], [], [], []
                ok = True
                for x, y in pairs:
                    z = tfn(x)
                    if mode == "factor":
                        if y.shape[0] % z.shape[0] or y.shape[1] % z.shape[1]:
                            ok = False
                            break
                        factors.append((y.shape[0] // z.shape[0], y.shape[1] // z.shape[1]))
                        target_shape = y.shape
                    else:
                        target_shape = y.shape
                        shapes.append(target_shape)
                    zz = tile_to_shape(z, target_shape)
                    srcs.append(zz)
                    dsts.append(y)
                if not ok:
                    continue
                if mode == "factor" and len(set(factors)) != 1:
                    continue
                if mode == "constant" and len(set(shapes)) != 1:
                    continue
                cmap = infer_color_map(srcs, dsts)
                if cmap is None:
                    continue
                outs = []
                for t in tests:
                    z = tfn(t)
                    if mode == "factor":
                        fy, fx = factors[0]
                        target_shape = (z.shape[0] * fy, z.shape[1] * fx)
                    else:
                        target_shape = shapes[0]
                    outs.append(apply_color_map(tile_to_shape(z, target_shape), cmap))
                yield Candidate(f"periodic_tile:{mode}:{tname}", outs, 0.97)

    def _fit_shifted_tile(self, pairs, tests):
        factors = []
        srcs, dsts = [], []
        ok = True
        for x, y in pairs:
            if y.shape[0] % x.shape[0] or y.shape[1] % x.shape[1]:
                ok = False
                break
            fy, fx = y.shape[0] // x.shape[0], y.shape[1] // x.shape[1]
            factors.append((fy, fx))
            srcs.append(shifted_tile_to_shape(x, y.shape))
            dsts.append(y)
        if ok and len(set(factors)) == 1:
            cmap = infer_color_map(srcs, dsts)
            if cmap is not None:
                fy, fx = factors[0]
                outs = [apply_color_map(shifted_tile_to_shape(t, (t.shape[0] * fy, t.shape[1] * fx)), cmap)
                        for t in tests]
                yield Candidate(f"shifted_tile:{fy}x{fx}", outs, 0.985)

    def _masked_self_tile(self, a: np.ndarray, bg: int) -> np.ndarray:
        h, w = a.shape
        out = np.full((h * h, w * w), int(bg), dtype=np.int8)
        for y in range(h):
            for x in range(w):
                if int(a[y, x]) != int(bg):
                    out[y * h:(y + 1) * h, x * w:(x + 1) * w] = a
        return out

    def _fit_masked_self_tile(self, pairs, tests):
        for bg_mode in ("zero", "major"):
            srcs, dsts = [], []
            ok = True
            for x, y in pairs:
                bg = 0 if bg_mode == "zero" else bg_color(x)
                z = self._masked_self_tile(x, bg)
                if z.shape != y.shape:
                    ok = False
                    break
                srcs.append(z)
                dsts.append(y)
            if not ok:
                continue
            cmap = infer_color_map(srcs, dsts)
            if cmap is None:
                continue
            outs = []
            for t in tests:
                bg = 0 if bg_mode == "zero" else bg_color(t)
                outs.append(apply_color_map(self._masked_self_tile(t, bg), cmap))
            yield Candidate(f"masked_self_tile:{bg_mode}", outs, 0.99)

    def _split_by_separator(self, a: np.ndarray):
        h, w = a.shape
        for x in range(1, w - 1):
            if len(set(a[:, x].tolist())) == 1 and x == w - x - 1:
                yield "v", int(a[0, x]), a[:, :x], a[:, x + 1:]
        for y in range(1, h - 1):
            if len(set(a[y, :].tolist())) == 1 and y == h - y - 1:
                yield "h", int(a[y, 0]), a[:y, :], a[y + 1:, :]

    def _overlay_panels(self, left: np.ndarray, right: np.ndarray, mode: str) -> Optional[np.ndarray]:
        if left.shape != right.shape:
            return None
        vals = set(left.ravel().tolist()) | set(right.ravel().tolist())
        bg = 0 if 0 in vals else bg_color(np.concatenate([left.reshape(-1), right.reshape(-1)]))
        out = np.full(left.shape, int(bg), dtype=np.int8)
        if mode == "intersection_same":
            mask = (left == right) & (left != bg)
            out[mask] = left[mask]
        elif mode == "intersection_nonbg":
            mask = (left != bg) & (right != bg)
            out[mask] = left[mask]
        elif mode == "left_minus_right":
            mask = (left != bg) & (right == bg)
            out[mask] = left[mask]
        elif mode == "right_minus_left":
            mask = (right != bg) & (left == bg)
            out[mask] = right[mask]
        elif mode == "xor":
            mask = (left != bg) ^ (right != bg)
            out[mask] = np.where(left[mask] != bg, left[mask], right[mask])
        elif mode == "binary_intersection_same":
            mask = (left == right) & (left != bg)
            out[mask] = 1
        elif mode == "binary_intersection_nonbg":
            mask = (left != bg) & (right != bg)
            out[mask] = 1
        elif mode == "binary_left_minus_right":
            mask = (left != bg) & (right == bg)
            out[mask] = 1
        elif mode == "binary_right_minus_left":
            mask = (right != bg) & (left == bg)
            out[mask] = 1
        elif mode == "binary_xor":
            mask = (left != bg) ^ (right != bg)
            out[mask] = 1
        else:
            return None
        return out

    def _fit_separator_overlay(self, pairs, tests):
        modes = [
            "intersection_same", "intersection_nonbg", "left_minus_right", "right_minus_left", "xor",
            "binary_intersection_same", "binary_intersection_nonbg", "binary_left_minus_right",
            "binary_right_minus_left", "binary_xor",
        ]
        for mode in modes:
            for side_order in ("normal", "swap"):
                srcs, dsts = [], []
                ok = True
                orientations = []
                for x, y in pairs:
                    splits = list(self._split_by_separator(x))
                    matched = False
                    for orient, _sep, a, b in splits:
                        left, right = (a, b) if side_order == "normal" else (b, a)
                        z = self._overlay_panels(left, right, mode)
                        if z is not None and z.shape == y.shape:
                            srcs.append(z)
                            dsts.append(y)
                            orientations.append(orient)
                            matched = True
                            break
                    if not matched:
                        ok = False
                        break
                if not ok or len(set(orientations)) > 1:
                    continue
                cmap = infer_color_map(srcs, dsts)
                if cmap is None:
                    continue
                outs = []
                for t in tests:
                    splits = [s for s in self._split_by_separator(t) if s[0] == orientations[0]]
                    if not splits:
                        outs.append(t.copy())
                        continue
                    _orient, _sep, a, b = splits[0]
                    left, right = (a, b) if side_order == "normal" else (b, a)
                    z = self._overlay_panels(left, right, mode)
                    outs.append(apply_color_map(z if z is not None else t, cmap))
                yield Candidate(f"separator:{mode}:{side_order}", outs, 0.94)

    def _fit_component_selector(self, pairs, tests):
        selectors = [
            ("largest", lambda comps: max(comps, key=lambda c: c["area"]) if comps else None),
            ("smallest", lambda comps: min(comps, key=lambda c: c["area"]) if comps else None),
            ("topmost", lambda comps: min(comps, key=lambda c: (c["bbox"][0], c["bbox"][2], -c["area"])) if comps else None),
            ("leftmost", lambda comps: min(comps, key=lambda c: (c["bbox"][2], c["bbox"][0], -c["area"])) if comps else None),
        ]
        for sname, selector in selectors:
            for mask in (None, 0):
                for tname, tfn in transforms():
                    srcs, dsts = [], []
                    ok = True
                    for x, y in pairs:
                        comp = selector(connected_components(x))
                        if comp is None:
                            ok = False
                            break
                        z = tfn(crop_component(x, comp, mask_bg=mask))
                        if z.shape != y.shape:
                            ok = False
                            break
                        srcs.append(z)
                        dsts.append(y)
                    if not ok:
                        continue
                    cmap = infer_color_map(srcs, dsts)
                    if cmap is None:
                        continue
                    outs = []
                    for t in tests:
                        comp = selector(connected_components(t))
                        if comp is None:
                            outs.append(t.copy())
                        else:
                            outs.append(apply_color_map(tfn(crop_component(t, comp, mask_bg=mask)), cmap))
                    yield Candidate(f"component:{sname}:{mask}:{tname}", outs, 0.93)

    def _fit_gridline_decomposition(self, pairs, tests):
        for mode in ("rows", "cols"):
            for reducer_name, reducer in (
                ("remove_uniform", self._remove_uniform_lines),
                ("keep_uniform", self._keep_uniform_lines),
            ):
                srcs, dsts = [], []
                ok = True
                for x, y in pairs:
                    z = reducer(x, mode)
                    if z is None or z.shape != y.shape:
                        ok = False
                        break
                    srcs.append(z)
                    dsts.append(y)
                if not ok:
                    continue
                cmap = infer_color_map(srcs, dsts)
                if cmap is None:
                    continue
                outs = []
                for t in tests:
                    z = reducer(t, mode)
                    outs.append(apply_color_map(z if z is not None else t, cmap))
                yield Candidate(f"gridline:{mode}:{reducer_name}", outs, 0.86)

    def _remove_uniform_lines(self, a: np.ndarray, mode: str) -> Optional[np.ndarray]:
        if mode == "rows":
            keep = [i for i in range(a.shape[0]) if len(set(a[i].tolist())) > 1]
            return a[keep, :].copy() if keep else None
        keep = [j for j in range(a.shape[1]) if len(set(a[:, j].tolist())) > 1]
        return a[:, keep].copy() if keep else None

    def _keep_uniform_lines(self, a: np.ndarray, mode: str) -> Optional[np.ndarray]:
        if mode == "rows":
            keep = [i for i in range(a.shape[0]) if len(set(a[i].tolist())) == 1]
            return a[keep, :].copy() if keep else None
        keep = [j for j in range(a.shape[1]) if len(set(a[:, j].tolist())) == 1]
        return a[:, keep].copy() if keep else None

    def _nearest_shape_output(self, pairs, tests):
        outs = [y for _, y in pairs]
        for t in tests:
            best = min(outs, key=lambda y: abs(y.shape[0] - t.shape[0]) + abs(y.shape[1] - t.shape[1]))
            yield Candidate("nearest_train_output", [best.copy() for _ in tests], 0.01)


def solve_challenges(challenges: dict) -> dict:
    solver = Arc2SymbolicSolver()
    submission = {}
    for task_id, task in challenges.items():
        solved = solver.solve_task(task)
        submission[task_id] = []
        for attempts in solved:
            submission[task_id].append(
                {
                    "attempt_1": np.asarray(attempts[0], dtype=int).tolist(),
                    "attempt_2": np.asarray(attempts[1], dtype=int).tolist(),
                }
            )
    return submission


def score_submission(challenges: dict, solutions: dict, submission: dict) -> Tuple[int, int, List[str]]:
    correct = 0
    total = 0
    solved_ids = []
    for task_id, answers in solutions.items():
        task_ok = False
        for i, truth in enumerate(answers):
            total += 1
            pred = submission.get(task_id, [{}])[i]
            t = to_grid(truth)
            if to_grid(pred.get("attempt_1", [[0]])) == t or to_grid(pred.get("attempt_2", [[0]])) == t:
                correct += 1
                task_ok = True
        if task_ok:
            solved_ids.append(task_id)
    return correct, total, solved_ids


def load_json(path: str | Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    def _find_arc_file(fname):
        for base in ["/kaggle/input/arc-prize-2026-arc-agi-2", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"]:
            p = os.path.join(base, fname)
            if os.path.exists(p):
                return p
        return os.path.join("/kaggle/input/competitions/arc-prize-2026-arc-agi-2", fname)

    def _find_model_dir():
        candidates = [
            "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
            "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
            "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
            "/kaggle/input/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
        ]
        for c in candidates:
            if os.path.isfile(os.path.join(c, "config.json")):
                return c
        return "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"


    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=2,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=_find_model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.16)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=24, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    keys_all = sorted(data.keys())
    if not rerun_mode:
        keys_all = keys_all[:16]
    for key in keys_all:
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}


In [ ]:
import os
import json
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder
from arc2_symbolic import Arc2SymbolicSolver

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

def _find_arc_file(fname):
    for base in ["/kaggle/input/competitions/arc-prize-2026-arc-agi-2", "/kaggle/input/arc-prize-2026-arc-agi-2"]:
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return os.path.join("/kaggle/input/competitions/arc-prize-2026-arc-agi-2", fname)

if rerun_mode:
    test_path = _find_arc_file("arc-agi_test_challenges.json")
    data = ArcDataset.from_file(test_path)
else:
    test_path = _find_arc_file("arc-agi_evaluation_challenges.json")
    sol_path = _find_arc_file("arc-agi_evaluation_solutions.json")
    data = ArcDataset.from_file(test_path)
    if os.path.exists(sol_path):
        data = data.load_replies(sol_path)

# =========================================================================
# 1. Neural Decoding Selection (Exact 33.89 LB Pipeline)
# =========================================================================
decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

if os.path.isdir("/kaggle/inference_outputs"):
    decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo())
print(f"[Neural Pipeline] Generated baseline submission for {len(submission)} tasks.")

# =========================================================================
# 2. Monotonic System 2 Certified Geometric Program Synthesis
# =========================================================================
print("[Symbolic Engine] Running certified geometric program synthesis...")
symbolic_solver = Arc2SymbolicSolver()

verified_injected = 0
fallbacks_repaired = 0
raw_queries = getattr(data, 'queries', {})

def is_dummy_or_invalid(grid):
    if not isinstance(grid, list) or len(grid) == 0:
        return True
    if grid == [[0]]:
        return True
    h = len(grid)
    if h <= 0 or h > 30:
        return True
    if not isinstance(grid[0], list) or len(grid[0]) == 0:
        return True
    w = len(grid[0])
    if w <= 0 or w > 30:
        return True
    for row in grid:
        if not isinstance(row, list) or len(row) != w:
            return True
        for cell in row:
            if not isinstance(cell, int) or cell < 0 or cell > 9:
                return True
    return False

for task_id in list(data.keys):
    task_dict = raw_queries.get(task_id, {})
    test_cases = task_dict.get('test', [])
    
    verified_result = None
    try:
        verified_result = symbolic_solver.solve_verified_geometry(task_dict)
    except Exception:
        pass

    preds = submission.get(task_id, [])
    for test_idx, test_case in enumerate(test_cases):
        test_in = test_case.get('input', [[0]])
        test_in_arr = np.array(test_in)

        if test_idx >= len(preds):
            preds.append({'attempt_1': [[0]], 'attempt_2': [[0]]})
            submission[task_id] = preds

        target_pred = preds[test_idx]

        # Monotonic Certified Symbolic Program Injection:
        # 1. If attempt_1 is dummy/missing, inject certified program into attempt_1!
        # 2. If attempt_1 has a valid neural prediction, but attempt_2 is dummy or duplicate,
        #    inject certified program into attempt_2 (preserving neural attempt_1)!
        if verified_result is not None:
            prog_name, sym_preds = verified_result
            if test_idx < len(sym_preds):
                sym_grid = sym_preds[test_idx].astype(int).tolist()
                if not is_dummy_or_invalid(sym_grid):
                    cur_att1 = target_pred.get('attempt_1')
                    cur_att2 = target_pred.get('attempt_2')
                    if is_dummy_or_invalid(cur_att1):
                        target_pred['attempt_1'] = sym_grid
                        verified_injected += 1
                        print(f"[Verified Inject] {task_id}_{test_idx} -> attempt_1 via {prog_name}")
                    elif is_dummy_or_invalid(cur_att2) or cur_att2 == cur_att1:
                        if sym_grid != cur_att1:
                            target_pred['attempt_2'] = sym_grid
                            verified_injected += 1
                            print(f"[Verified Inject] {task_id}_{test_idx} -> attempt_2 via {prog_name}")

        # Smart Fallbacks for any remaining dummy/duplicate attempts
        cur_att1 = target_pred.get('attempt_1')
        if is_dummy_or_invalid(cur_att1):
            target_pred['attempt_1'] = [r[:] for r in test_in]
            fallbacks_repaired += 1

        cur_att2 = target_pred.get('attempt_2')
        if is_dummy_or_invalid(cur_att2) or cur_att2 == target_pred.get('attempt_1'):
            fallbacks_repaired += 1
            cand = np.rot90(test_in_arr, 1).tolist()
            if not is_dummy_or_invalid(cand) and cand != target_pred.get('attempt_1'):
                target_pred['attempt_2'] = cand
            else:
                cand2 = np.fliplr(test_in_arr).tolist()
                if not is_dummy_or_invalid(cand2) and cand2 != target_pred.get('attempt_1'):
                    target_pred['attempt_2'] = cand2
                else:
                    cand3 = [r[:] for r in target_pred.get('attempt_1')]
                    cand3[0][0] = (cand3[0][0] + 1) % 10
                    target_pred['attempt_2'] = cand3

print(f"[Monotonic Arbitration] Certified injected: {verified_injected}, Fallbacks repaired: {fallbacks_repaired}")

# =========================================================================
# 3. Atomic Serialization of submission.json
# =========================================================================
with open("submission.json.tmp", "w") as f:
    json.dump(submission, f, separators=(',', ':'))
os.replace("submission.json.tmp", "submission.json")
print(f"[Submission] Successfully serialized submission.json ({os.path.getsize('submission.json')} bytes).")

if not rerun_mode:
    try:
        decoder.benchmark_selection_algos()
    except Exception as e:
        print("benchmark failed:", repr(e))
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))
